# **MOVIE REVIEW CLASSIFICATION USING LLMs**

This notebook demonstrates the practical differences between encoder-only and decoder-only transformer architectures through a classification task.

**KEY CONCEPTS BEING COMPARED:**

1) **Encoder-only models (BERT)**: Designed for understanding tasks, fine-tuned for classification

2) **Decoder-only models (Phi-3)**: Originally for generation, adapted for classification using proper chat templates

3) **Fine-tuning vs Prompt Engineering**: Comparing supervised learning against instruction-following approaches

**WHY THIS COMPARISON MATTERS:**
- Shows when to use fine-tuning vs prompting
- Demonstrates architectural differences in practice
- Helps choose the right approach based on data availability and computational resources

## **Library Imports and Environment Setup**

**Purpose**: Import all necessary libraries and configure the environment for stable model training.

**Key Components:**
- **PyTorch & TensorFlow**: Deep learning frameworks
- **HuggingFace Transformers**: Pre-trained model library (BERT, Phi-3)
- **Datasets**: Efficient data handling
- **scikit-learn**: Evaluation metrics
- **Environment Variables**: Prevent common issues during training

In [8]:
import os
import time
import random
import warnings

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# Keep warnings visible but only show each one once (less noise than fully
# silencing them, which can hide useful deprecation notices).
warnings.filterwarnings("once")

# Reproducibility: fix all the random seeds we rely on.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Environment settings that reduce noise / avoid common Colab issues
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"


Using device: cuda


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## **Data Import**

**Dataset**: IMDB movie review dataset for binary sentiment classification (positive/negative)

**Why IMDB**:
- Well-established benchmark for sentiment analysis
- Large dataset (50K reviews) but we use a subset (5K train, 1K test) for fast training
- Binary classification makes it easy to compare model performance

**Data Processing Steps:**
1. Load the **raw-text** IMDB dataset directly from the Hugging Face Hub
2. Shuffle with a fixed seed and select small, reproducible subsets
3. Use the **same** test subset for both models so the comparison is fair


In [9]:
# Load the IMDB sentiment dataset (raw text) directly from the Hugging Face Hub.
# This gives us the original review text with casing and punctuation intact,
# which is what we actually want to feed to the models. It also avoids the
# fragile word-index -> text decoding that the Keras version requires.
print("Loading the IMDB dataset from the Hugging Face Hub...")
imdb = load_dataset("stanfordnlp/imdb")

# IMDB ships 25k train / 25k test, ordered by label, so shuffling is essential.
# We use a fixed seed and take small balanced subsets for a fast comparison.
train_data = imdb["train"].shuffle(seed=SEED).select(range(5000))
test_data = imdb["test"].shuffle(seed=SEED).select(range(1000))

print(f"\u2705 Loaded IMDB. Training samples: {len(train_data)}, "
      f"Test samples: {len(test_data)}")

# Display a couple of sample reviews
print("\nSample reviews:")
for i in range(2):
    text = train_data[i]["text"]
    preview = text[:200] + "..." if len(text) > 200 else text
    label = train_data[i]["label"]
    print(f"Text: {preview}")
    print(f"Label: {label} ({'Positive' if label == 1 else 'Negative'})")
    print("-" * 50)

# Label distribution in the training subset
labels = np.array(train_data["label"])
print("\nLabel distribution in training set:")
print(f"Negative (0): {np.sum(labels == 0)} samples")
print(f"Positive (1): {np.sum(labels == 1)} samples")


Loading the IMDB dataset from the Hugging Face Hub...
✅ Loaded IMDB. Training samples: 5000, Test samples: 1000

Sample reviews:
Text: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: 1 (Positive)
--------------------------------------------------
Text: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub y...
Label: 1 (Positive)
--------------------------------------------------

Label distribution in training set:
Negative (0): 2494 samples
Positive (1): 2506 samples


## **BERT MODEL**

**Architecture**: Encoder-only transformer designed specifically for understanding tasks

**Key Characteristics:**
- **Bidirectional attention**: Can see both past and future tokens simultaneously
- **Masked Language Modeling**: Pre-trained to predict masked tokens in sentences
- **Classification head**: Added on top for downstream sentiment classification
- **Fine-tuning approach**: Requires labeled data and parameter updates

**Model Choice**: `bert-base-uncased`
- 110M parameters
- Uncased = doesn't distinguish between uppercase/lowercase
- Base size balances performance with training speed

**Fine-tuning Concept**:
- Start with pre-trained BERT weights
- Add classification head (2 classes: positive/negative)
- Update model parameters using labeled sentiment data
- This adapts BERT's general language understanding to sentiment classification

**Expected Outcome**: BERT learns to map movie review text to sentiment labels

In [10]:
# Initialize BERT model and tokenizer
model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
).to(device)

# Freeze the BERT backbone; train only the classification head.
for param in bert_model.bert.parameters():
    param.requires_grad = False

print(f"BERT model parameters: {bert_model.num_parameters():,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model parameters: 109,483,778


## **Data Tokenization for BERT**

**Purpose**: Convert text into numerical format that BERT can process

**Tokenization Process:**
1. **Text → Tokens**: Split text into subwords using BERT's vocabulary
2. **Tokens → IDs**: Convert tokens to numerical indices
3. **Dynamic padding**: Each batch is padded to its own longest sequence at run time (via a data collator) — faster than padding every example to 512
4. **Attention Masks**: Tell the model which tokens are real vs padding

**Why truncate at 512 tokens**: 512 is BERT's maximum sequence length.

In [11]:
# Tokenize with the HuggingFace Datasets .map() API. We DON'T pad here:
# padding is done per-batch by the data collator below (dynamic padding).
def tokenize_for_bert(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=512)

print("Tokenizing datasets for BERT...")
bert_train_ds = train_data.map(tokenize_for_bert, batched=True, remove_columns=["text"])
bert_test_ds = test_data.map(tokenize_for_bert, batched=True, remove_columns=["text"])

# The Trainer expects the target column to be named "labels".
bert_train_ds = bert_train_ds.rename_column("label", "labels")
bert_test_ds = bert_test_ds.rename_column("label", "labels")

# NOTE: we deliberately do NOT call .set_format("torch"). The data collator
# converts each batch to tensors, and skipping the datasets "torch" formatter
# avoids a torchvision `VideoReader` import bug present on some Colab images.
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

print("\u2705 Tokenization complete!")
print(f"Training dataset size: {len(bert_train_ds)}")
print(f"Test dataset size: {len(bert_test_ds)}")

Tokenizing datasets for BERT...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Tokenization complete!
Training dataset size: 5000
Test dataset size: 1000


## **BERT Fine-tuning Process**


**Training Configuration:**
- **1 epoch**: Sufficient for demonstration (more epochs = better performance)
- **Batch size 8**: Memory-efficient for Colab
- **Learning rate**: Automatically handled by Trainer
- **Evaluation**: Monitor accuracy during training



In [12]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=50,          # ~625 steps total for 1 epoch, so keep warmup short
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",    # NOTE: older transformers used `evaluation_strategy`
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=[],             # Disable wandb and other experiment tracking
)

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Initialize trainer (the data collator handles per-batch padding)
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_ds,
    eval_dataset=bert_test_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# Train the model
print("Starting BERT fine-tuning...")
trainer.train()

# Save the model
trainer.save_model("./bert-imdb-final")

Starting BERT fine-tuning...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.324904,0.290370,0.903000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## **BERT Model Evaluation**

**Evaluation Metrics:**
- **Accuracy**: Overall percentage of correct predictions
- **Precision/Recall**: Detailed per-class performance
- **F1-Score**: Harmonic mean of precision and recall

**Classification Report**:
- Shows how well BERT distinguishes positive vs negative reviews
- Identifies any bias toward one class
- Provides confidence in model's sentiment predictions

**Expected Performance**: BERT typically achieves 85-90% accuracy on IMDB sentiment classification

In [13]:
import gc

# Evaluate BERT
bert_results = trainer.evaluate()
print("BERT Results:")
print(f"Accuracy: {bert_results['eval_accuracy']:.4f}")

# Cache predictions and true labels NOW, so we can compare against Phi-3 later
# without keeping the BERT model in GPU memory at the same time.
bert_pred_output = trainer.predict(bert_test_ds)
bert_predictions = np.argmax(bert_pred_output.predictions, axis=1)
true_labels = np.array(bert_test_ds["labels"])

print("\nClassification Report:")
print(classification_report(true_labels, bert_predictions,
                            target_names=["Negative", "Positive"]))

# Free GPU memory before loading the much larger Phi-3 model. Both models do
# not need to be resident at once (T4 Colab GPUs only have ~15 GB).
del trainer, bert_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Training Loss,Validation Loss,Epoch,Accuracy
0.324904,0.290370,1,0.903000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


BERT Results:
Accuracy: 0.9030


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Classification Report:
              precision    recall  f1-score   support

    Negative       0.94      0.87      0.90       512
    Positive       0.87      0.94      0.90       488

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.91      0.90      0.90      1000



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## **Phi-3 Instruct Model Loading**

**Architecture**: Decoder-only transformer (like GPT) but instruction-tuned

**Key Differences from BERT:**
- **Generative model**: Originally designed to generate text, not classify
- **Autoregressive**: Processes tokens left-to-right (not bidirectional)
- **Instruction-tuned**: Fine-tuned to follow chat-style instructions
- **Much larger**: 3.8B parameters vs BERT's 110M

**Model Choice**: `microsoft/Phi-3-mini-4k-instruct`
- Instruction-tuned version (can follow prompts)
- 4K context length
- Optimized for chat-style interactions

**Approach**: Instead of fine-tuning, we'll use prompting to classify sentiment

> **Important detail — left padding:** Decoder-only models must use **left** padding for batched generation, because generation continues from the last real token. We set `padding_side = "left"` explicitly below. (BERT, an encoder, is happy with the default right padding.)

In [14]:
# Load instruct model
instruct_model_name = "microsoft/Phi-3-mini-4k-instruct"
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Set a padding token if there isn't one.
if instruct_tokenizer.pad_token is None:
    instruct_tokenizer.pad_token = instruct_tokenizer.eos_token

# IMPORTANT: decoder-only models need LEFT padding for correct *batched*
# generation. Generation continues from the last real token, so padding must
# sit on the left. We set this explicitly rather than relying on the default.
instruct_tokenizer.padding_side = "left"

print(f"Instruct model parameters: {instruct_model.num_parameters():,}")


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1856: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db81e10>


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db80600>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db80590>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db804b0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db802f0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db80280>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db801a0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db80050>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db819b0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db814e0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db81400>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7dbb4db807c0>


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Instruct model parameters: 3,821,079,552


## **Instruction-tuned Classification with Phi-3**

**Prompting Approach**:
- No parameter updates or fine-tuning required
- Use chat template to ask model to classify sentiment
- Model uses its pre-trained knowledge + instruction-following ability

**Chat Template Structure:**
- **System**: Define the model's role (sentiment analyzer)
- **User**: Provide the review and ask for classification
- **Assistant**: Model generates "positive" or "negative"

**Key Advantages:**
- **Zero-shot**: Works without any training data
- **Fast**: No training time required
- **Flexible**: Can easily change task by modifying prompt

**Key Challenges:**
- **Slower inference**: Must generate tokens vs simple classification
- **Less precise**: May generate unexpected responses
- **Larger memory**: 3.8B parameters vs 110M for BERT

In [15]:
def classify_with_instruction_model(texts, model, tokenizer, batch_size=8):
    """Classify sentiment with an instruction-tuned model using the tokenizer's
    chat template and real (left-padded) batching."""
    predictions = []
    start_time = time.time()
    total = len(texts)

    print(f"\U0001f680 Classifying {total} texts (batch size: {batch_size})...")

    for i in range(0, total, batch_size):
        batch_texts = texts[i:i + batch_size]

        # Build a properly chat-formatted prompt for each review using the
        # model's own chat template (more robust than hand-writing the tags).
        prompts = []
        for text in batch_texts:
            messages = [
                {"role": "system",
                 "content": "You are a helpful assistant that analyzes movie review sentiment."},
                {"role": "user",
                 "content": ('Analyze the sentiment of this movie review and respond with '
                             'exactly one word: either "positive" or "negative".\n\n'
                             f'Movie Review: "{text[:512]}"')},
            ]
            prompts.append(
                tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            )

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        # With LEFT padding, the prompt fills the first `input_len` columns for
        # every row, so the freshly generated tokens are everything after it.
        input_len = inputs["input_ids"].shape[1]
        for output in outputs:
            response = tokenizer.decode(
                output[input_len:], skip_special_tokens=True
            ).strip().lower()

            if "positive" in response:
                predictions.append(1)
            elif "negative" in response:
                predictions.append(0)
            else:
                # Fallback when the model says something off-format. Defaulting
                # to one class introduces a small bias; in production you might
                # re-prompt or constrain the output instead.
                predictions.append(0)
                if response:
                    print(f"\n\u26a0\ufe0f Unclear response: '{response[:50]}'")

        processed = min(i + batch_size, total)
        if processed % (batch_size * 5) == 0 or processed == total:
            elapsed = time.time() - start_time
            eta = elapsed / processed * (total - processed)
            print(f"\U0001f4ca {processed}/{total} ({processed / total * 100:.1f}%) | "
                  f"{elapsed / processed:.2f}s/review | ETA: {eta:.0f}s")

    elapsed = time.time() - start_time
    print(f"\n\u2705 Done in {elapsed:.1f}s ({elapsed / total:.2f}s/review)")
    return np.array(predictions)


# IMPORTANT: evaluate Phi-3 on the SAME test reviews BERT was evaluated on.
test_texts = test_data["text"]
test_true = np.array(test_data["label"])

instruction_predictions = classify_with_instruction_model(
    test_texts, instruct_model, instruct_tokenizer, batch_size=8
)

instruction_accuracy = accuracy_score(test_true, instruction_predictions)
print(f"\n\U0001f3af Instruction-tuned accuracy ({len(test_texts)} samples): "
      f"{instruction_accuracy:.4f}")

# Show some example predictions
print("\n\U0001f4cb Example predictions:")
for i in range(3):
    preview = test_texts[i][:100] + "..." if len(test_texts[i]) > 100 else test_texts[i]
    mark = "\u2705" if test_true[i] == instruction_predictions[i] else "\u274c"
    true_label = "Positive" if test_true[i] == 1 else "Negative"
    pred_label = "Positive" if instruction_predictions[i] == 1 else "Negative"
    print(f"{mark} True: {true_label}, Predicted: {pred_label}")
    print(f"   Text: {preview}")
    print("-" * 50)


🚀 Classifying 1000 texts (batch size: 8)...
📊 40/1000 (4.0%) | 0.12s/review | ETA: 119s
📊 80/1000 (8.0%) | 0.12s/review | ETA: 112s
📊 120/1000 (12.0%) | 0.12s/review | ETA: 104s
📊 160/1000 (16.0%) | 0.12s/review | ETA: 99s
📊 200/1000 (20.0%) | 0.12s/review | ETA: 95s
📊 240/1000 (24.0%) | 0.12s/review | ETA: 91s
📊 280/1000 (28.0%) | 0.12s/review | ETA: 88s
📊 320/1000 (32.0%) | 0.12s/review | ETA: 83s
📊 360/1000 (36.0%) | 0.12s/review | ETA: 78s
📊 400/1000 (40.0%) | 0.12s/review | ETA: 73s
📊 440/1000 (44.0%) | 0.12s/review | ETA: 68s
📊 480/1000 (48.0%) | 0.12s/review | ETA: 62s
📊 520/1000 (52.0%) | 0.12s/review | ETA: 57s
📊 560/1000 (56.0%) | 0.12s/review | ETA: 52s
📊 600/1000 (60.0%) | 0.12s/review | ETA: 47s
📊 640/1000 (64.0%) | 0.12s/review | ETA: 42s
📊 680/1000 (68.0%) | 0.12s/review | ETA: 37s
📊 720/1000 (72.0%) | 0.12s/review | ETA: 33s
📊 760/1000 (76.0%) | 0.12s/review | ETA: 28s
📊 800/1000 (80.0%) | 0.12s/review | ETA: 23s
📊 840/1000 (84.0%) | 0.12s/review | ETA: 19s
📊 880/1000 (

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## **Model Comparison and Key Insights**

**Performance Summary (typical):**
- **BERT (Fine-tuned)**: around 90% accuracy
- **Phi-3 (Instruction-tuned)**: around 89% accuracy (zero-shot)

Both numbers come from the **same** 1000-review test set, so the gap between them is meaningful.

**Architecture Trade-offs:**

**BERT (Encoder-only) Strengths:**
- Strong accuracy on classification tasks
- Faster inference (single forward pass)
- Smaller memory footprint
- Bidirectional context understanding

**Phi-3 (Decoder-only) Strengths:**
- No training data required (zero-shot)
- Extremely flexible (any NLP task via prompting)
- Better few-shot learning capabilities
- Can explain reasoning in natural language

**When to Choose Each:**
- **BERT**: When you have labeled data, need high accuracy, want fast inference
- **Phi-3**: When you lack training data, need flexibility, want to rapidly prototype

**Real-world Implications:**
- Fine-tuning requires ML expertise and computational resources
- Prompting is accessible to non-ML experts but may need more powerful hardware
- Consider data availability, accuracy requirements, and deployment constraints